# Build the immutable Llava Onevision 7B snapshot

Settings: **Accelerator OFF**, **Internet ON**. Attach the authenticated CertVIC CODE bundle under any account, title, filename, extension, mount, or nesting. Run All without editing. Canonical output: `llava_onevision_7b_snapshot.zip`. This builder streams Hub bytes directly into one ZIP64 archive, never installs into the notebook kernel, and never retains a second full snapshot ZIP.


In [ ]:
import json, platform, sys

print(json.dumps({
    "status": "IMMEDIATE_SNAPSHOT_PROVISIONING_PROBE",
    "executable": sys.executable,
    "implementation": platform.python_implementation(),
    "python": platform.python_version(),
    "architecture": platform.machine(),
    "system": platform.system(),
    "libc": platform.libc_ver(),
}, indent=2))
if platform.python_implementation() != "CPython" or not platform.python_version().startswith("3.12."):
    raise RuntimeError("CERTVIC_RUNTIME_01_PYTHON_PROFILE_NOT_SUPPORTED: CPython 3.12 required")
if platform.system() != "Linux" or platform.machine().lower() != "x86_64":
    raise RuntimeError("CERTVIC_RUNTIME_01_PYTHON_PROFILE_NOT_SUPPORTED: Linux x86_64 required")

import hashlib, json, os, pathlib, shutil, stat, sys, zipfile

DISCOVERY_ERRORS = {
    "missing": "CERTVIC_DISCOVERY_01_REQUIRED_ROLE_NOT_FOUND",
    "ambiguous": "CERTVIC_DISCOVERY_02_AMBIGUOUS_DISTINCT_CONTENT",
    "authentication": "CERTVIC_DISCOVERY_03_CONTENT_AUTHENTICATION_FAILED",
}
DISCOVERY_POLICY = "CONTENT_AUTHENTICATED_ANY_LOCATION"
OPERATIONAL_FIELDS = {
    "builder_command", "created_time", "expected_kaggle_dataset_slug", "mount_path",
    "required_notebook", "validation_command",
}

def early_sha256(path):
    digest = hashlib.sha256()
    with pathlib.Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

def early_safe_member(info):
    name = info.filename
    normalized = name.replace("\\", "/")
    value = pathlib.PurePosixPath(normalized)
    mode = (info.external_attr >> 16) & 0xFFFF
    if (not normalized or normalized != name or normalized.endswith("/") or value.is_absolute()
            or ".." in value.parts or "." in value.parts or normalized.startswith("~")
            or "\x00" in normalized or info.is_dir() or stat.S_ISLNK(mode)
            or (mode and not stat.S_ISREG(mode))):
        raise RuntimeError(f"{DISCOVERY_ERRORS['authentication']}: unsafe member {name!r}")
    return value.as_posix()

def early_content_identity(manifest, hash_files):
    identity_manifest = {key: value for key, value in manifest.items()
                         if key not in OPERATIONAL_FIELDS}
    identity_files = {name: record for name, record in hash_files.items()
                      if name not in {"README.md", "bundle_manifest.json"}}
    payload = json.dumps({"manifest": identity_manifest, "files": identity_files},
                         indent=2, sort_keys=True).encode() + b"\n"
    return hashlib.sha256(payload).hexdigest()

def early_verify_archive(path):
    with zipfile.ZipFile(path) as archive:
        infos = archive.infolist()
        names = [early_safe_member(info) for info in infos]
        if "bundle_manifest.json" not in names or "hash_manifest.json" not in names:
            return None
        if len(names) != len(set(names)) or archive.testzip() is not None:
            raise RuntimeError(f"{DISCOVERY_ERRORS['authentication']}: duplicate or corrupt archive")
        manifest_bytes = archive.read("bundle_manifest.json")
        hash_bytes = archive.read("hash_manifest.json")
        if len(manifest_bytes) > 8 * 1024 * 1024 or len(hash_bytes) > 8 * 1024 * 1024:
            raise RuntimeError(f"{DISCOVERY_ERRORS['authentication']}: oversized manifest")
        manifest = json.loads(manifest_bytes)
        hashes = json.loads(hash_bytes)
        if (manifest.get("schema") != "certvic.kaggle.bundle.v1"
                or hashes.get("schema") != "certvic.kaggle.hash_manifest.v1"
                or manifest.get("bundle_type") != "CODE"):
            return None
        declared, hash_files = manifest.get("files", {}), hashes.get("files", {})
        if (set(names) != set(hash_files) | {"hash_manifest.json"}
                or set(declared) != set(names) - {"bundle_manifest.json", "hash_manifest.json"}):
            raise RuntimeError(f"{DISCOVERY_ERRORS['authentication']}: file universe mismatch")
        for name, record in hash_files.items():
            payload = archive.read(name)
            observed = {"size": len(payload), "sha256": hashlib.sha256(payload).hexdigest()}
            if record != observed or (name in declared and declared[name] != observed):
                raise RuntimeError(f"{DISCOVERY_ERRORS['authentication']}: byte mismatch {name}")
        return manifest, hash_files, hashlib.sha256(manifest_bytes).hexdigest()

def early_verify_directory(path):
    root = pathlib.Path(path).resolve()
    manifest_path, hash_path = root / "bundle_manifest.json", root / "hash_manifest.json"
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    hashes = json.loads(hash_path.read_text(encoding="utf-8"))
    if (manifest.get("schema") != "certvic.kaggle.bundle.v1"
            or hashes.get("schema") != "certvic.kaggle.hash_manifest.v1"
            or manifest.get("bundle_type") != "CODE"):
        return None
    observed = {}
    for member in root.rglob("*"):
        mode = member.lstat().st_mode
        if member.is_symlink() or not (stat.S_ISDIR(mode) or stat.S_ISREG(mode)):
            raise RuntimeError(f"{DISCOVERY_ERRORS['authentication']}: unsafe extracted member")
        if stat.S_ISREG(mode):
            observed[member.relative_to(root).as_posix()] = member
    declared, hash_files = manifest.get("files", {}), hashes.get("files", {})
    if (set(observed) != set(hash_files) | {"hash_manifest.json"}
            or set(declared) != set(observed) - {"bundle_manifest.json", "hash_manifest.json"}):
        raise RuntimeError(f"{DISCOVERY_ERRORS['authentication']}: extracted universe mismatch")
    for name, record in hash_files.items():
        member = observed[name]
        actual = {"size": member.stat().st_size, "sha256": early_sha256(member)}
        if actual != record or (name in declared and declared[name] != actual):
            raise RuntimeError(f"{DISCOVERY_ERRORS['authentication']}: extracted byte mismatch {name}")
    return manifest, hash_files, early_sha256(manifest_path)

configured_roots = [value for value in os.environ.get("CERTVIC_INPUT_ROOTS", "").split(os.pathsep)
                    if value]
if not configured_roots:
    configured_roots = ["/kaggle/input", "/kaggle/working"]
INPUT_ROOTS = sorted({str(pathlib.Path(value).resolve()) for value in configured_roots
                      if pathlib.Path(value).is_dir() and not pathlib.Path(value).is_symlink()})
archive_candidates, directory_candidates = [], []
for root_value in INPUT_ROOTS:
    root = pathlib.Path(root_value)
    for current, directory_names, file_names in os.walk(root, followlinks=False):
        base = pathlib.Path(current)
        directory_names[:] = sorted(name for name in directory_names
                                     if not (base / name).is_symlink())
        if "bundle_manifest.json" in file_names and "hash_manifest.json" in file_names:
            directory_candidates.append(base.resolve())
        for name in sorted(file_names):
            candidate = base / name
            if candidate.is_symlink() or not candidate.is_file():
                continue
            try:
                with candidate.open("rb") as handle:
                    magic = handle.read(4)
            except OSError:
                continue
            if magic in {b"PK\x03\x04", b"PK\x05\x06", b"PK\x07\x08"}:
                archive_candidates.append(candidate.resolve())

valid, failures = [], []
for representation, candidates in (("zip_archive", sorted(set(archive_candidates))),
                                   ("extracted_directory", sorted(set(directory_candidates)))):
    for candidate in candidates:
        try:
            result = (early_verify_archive(candidate) if representation == "zip_archive"
                      else early_verify_directory(candidate))
        except (OSError, KeyError, json.JSONDecodeError, UnicodeDecodeError,
                zipfile.BadZipFile, RuntimeError) as error:
            failures.append(f"{candidate}: {error}")
            continue
        if result is None:
            continue
        manifest, hash_files, manifest_hash = result
        identity = early_content_identity(manifest, hash_files)
        expected = os.environ.get("CERTVIC_EXPECTED_CONTENT_ID_CODE")
        if expected and identity != expected.lower():
            failures.append(f"{candidate}: expected CODE content identity mismatch")
            continue
        valid.append({"path": candidate, "representation": representation,
                      "manifest": manifest, "manifest_sha256": manifest_hash,
                      "content_identity_sha256": identity})
if not valid:
    code = DISCOVERY_ERRORS["authentication"] if failures else DISCOVERY_ERRORS["missing"]
    raise RuntimeError(f"{code}: role=CODE failures={failures}")
identities = {row["content_identity_sha256"] for row in valid}
if len(identities) != 1:
    raise RuntimeError(f"{DISCOVERY_ERRORS['ambiguous']}: role=CODE candidates="
                       f"{[(row['content_identity_sha256'], str(row['path'])) for row in valid]}")
selected = min(valid, key=lambda row: os.path.normcase(str(row["path"])))
CODE_DISCOVERY_MIRRORS = sorted({str(row["path"]) for row in valid})
CODE_BUNDLE_SOURCE = str(selected["path"])
CODE_BUNDLE_HASH = selected["content_identity_sha256"]
CODE_ARCHIVE_SHA256 = (early_sha256(selected["path"])
                       if selected["representation"] == "zip_archive" else None)
if selected["representation"] == "zip_archive":
    CODE_EXTRACT_ROOT = pathlib.Path(os.environ.get(
        "CERTVIC_KAGGLE_WORKING_ROOT", "/kaggle/working")) / "certvic_code"
    if CODE_EXTRACT_ROOT.exists():
        if CODE_EXTRACT_ROOT.is_symlink() or not CODE_EXTRACT_ROOT.is_dir():
            raise RuntimeError(f"{DISCOVERY_ERRORS['authentication']}: unsafe CODE destination")
        shutil.rmtree(CODE_EXTRACT_ROOT)
    CODE_EXTRACT_ROOT.mkdir(parents=True)
    with zipfile.ZipFile(selected["path"]) as archive:
        for info in archive.infolist():
            name = early_safe_member(info)
            output = (CODE_EXTRACT_ROOT / name).resolve()
            output.relative_to(CODE_EXTRACT_ROOT.resolve())
            output.parent.mkdir(parents=True, exist_ok=True)
            with archive.open(info) as reader, output.open("xb") as writer:
                shutil.copyfileobj(reader, writer, length=1024 * 1024)
else:
    CODE_EXTRACT_ROOT = pathlib.Path(selected["path"])
CODE_BUNDLE_PATH = (CODE_BUNDLE_SOURCE if selected["representation"] == "zip_archive"
                    else str(CODE_EXTRACT_ROOT / "bundle_manifest.json"))
CODE_BUNDLE = CODE_BUNDLE_PATH
project_candidates = sorted(path.parent.resolve() for path in CODE_EXTRACT_ROOT.rglob("pyproject.toml")
                            if (path.parent / "certvic/__init__.py").is_file())
if len(project_candidates) != 1:
    raise RuntimeError(f"{DISCOVERY_ERRORS['authentication']}: CODE project root ambiguous")
PROJECT_ROOT = project_candidates[0]
sys.path.insert(0, str(PROJECT_ROOT))
from certvic.cvpr.content_discovery import (
    DISCOVERY_POLICY, discover_authenticated_input, resolve_content_bound_roles,
)
from certvic.cvpr.notebook_bootstrap import discover_unique_file, discover_unique_root
authenticated_code = discover_authenticated_input(
    "CODE", roots=INPUT_ROOTS, expected_identity=CODE_BUNDLE_HASH,
    materialization_root=pathlib.Path(os.environ.get(
        "CERTVIC_KAGGLE_WORKING_ROOT", "/kaggle/working")) / "certvic_authenticated_inputs",
)
if authenticated_code["content_identity_sha256"] != CODE_BUNDLE_HASH:
    raise RuntimeError(f"{DISCOVERY_ERRORS['authentication']}: early/shared CODE identity mismatch")
AUTHENTICATED_CONTENT_IDENTITIES = {"code_bundle": CODE_BUNDLE_HASH}
DISCOVERED_PROVENANCE = {"CODE": authenticated_code}
print({"discovery_policy": DISCOVERY_POLICY, "role": "CODE", "provider": None,
       "study": selected["manifest"].get("study"), "stage": selected["manifest"].get("stage"),
       "representation": selected["representation"], "discovered_path": CODE_BUNDLE_SOURCE,
       "content_identity_sha256": CODE_BUNDLE_HASH, "archive_sha256": CODE_ARCHIVE_SHA256,
       "mirrors": CODE_DISCOVERY_MIRRORS, "project_root": str(PROJECT_ROOT)})


In [ ]:
from pathlib import Path

PROVIDER = 'llava_onevision_7b'
MODEL_REPOSITORY = 'llava-hf/llava-onevision-qwen2-7b-ov-hf'
MODEL_COMMIT = '0d50680527681998e456c7b78950205bedd8a068'
PROCESSOR_COMMIT = MODEL_COMMIT
CANONICAL_OUTPUT = 'llava_onevision_7b_snapshot.zip'

from certvic.cvpr.snapshot_streaming_provisioner import (
    SnapshotStreamingError,
    stream_build_snapshot_bundle,
)

output = Path("/kaggle/working") / CANONICAL_OUTPUT
failure_report_path = Path("/kaggle/working") / (
    CANONICAL_OUTPUT.replace(".zip", "") + "_failure_report.json"
)
try:
    first = stream_build_snapshot_bundle(
        PROVIDER,
        output=output,
        working_dir=Path("/kaggle/working"),
        deps_dir=Path("/kaggle/working/certvic_snapshot_deps"),
        model_commit=MODEL_COMMIT,
        processor_commit=PROCESSOR_COMMIT,
    )
except SnapshotStreamingError as error:
    failure_report_path.write_text(
        json.dumps(error.report, indent=2, sort_keys=True) + "\n", encoding="utf-8"
    )
    output.unlink(missing_ok=True)
    print(json.dumps(error.report, indent=2, sort_keys=True))
    raise RuntimeError(
        f"{error.code}: failure_report={failure_report_path}"
    ) from error
except Exception as error:
    report = {
        "schema": "certvic.cvpr.snapshot_failure_report.v1",
        "status": "CERTVIC_SNAPSHOT_PROVISIONING_FAILED",
        "provider": PROVIDER,
        "error": f"{type(error).__name__}: {error}",
        "remediation": (
            "Inspect the streamed single-pass builder failure. Do not keep a raw snapshot tree "
            "or a second full ZIP beside the canonical archive."
        ),
        "paper_evidence": False,
    }
    failure_report_path.write_text(
        json.dumps(report, indent=2, sort_keys=True) + "\n", encoding="utf-8"
    )
    output.unlink(missing_ok=True)
    print(json.dumps(report, indent=2, sort_keys=True))
    raise RuntimeError(
        f"{report['status']}: failure_report={failure_report_path}"
    ) from error
print(json.dumps({
    "status": first["status"],
    "provider": PROVIDER,
    "model_repository": MODEL_REPOSITORY,
    "model_commit": MODEL_COMMIT,
    "processor_commit": PROCESSOR_COMMIT,
    "canonical_output": str(output),
    "size": output.stat().st_size,
    "sha256": first["sha256"],
    "determinism_proof": first["determinism_proof"],
    "raw_snapshot_retained": first["raw_snapshot_retained"],
    "second_full_zip_created": first["second_full_zip_created"],
    "dependency_isolation": first.get("dependency_isolation"),
    "disk_preflight": first.get("disk_preflight"),
    "paper_evidence": False,
}, indent=2, sort_keys=True))
print(
    "NEXT: download " + CANONICAL_OUTPUT + ", import it unchanged, then run the matching "
    "provider-specific 00B notebook with Accelerator OFF and Internet OFF."
)
